In [ ]:
# Allows the modification of the files imported without having to restart kernel or re-import them
%load_ext autoreload
%autoreload 2

In [ ]:
%config InlineBackend.figure_format='retina'
from IPython import display
display.display(display.HTML("<style>.container { width:95% !important; }</style>"))

# to have interactive plots : pip install ipympl
%matplotlib ipympl

### General imports
import sys
import glob
import numpy as np
import matplotlib.pyplot as plt
import healpy as hp
import pickle
import fitsio
from scipy.interpolate import LinearNDInterpolator

plt.rc('figure',figsize=(10,6))
plt.rc('font',size=16)

### Astropy configuration
import astropy.units as u

In [ ]:
t_start = {"h": 23, "m": 36}
duration_max_dict = {"h": 5, "m": 20}
duration_max = duration_max_dict["h"]*3600 + duration_max_dict["m"]*60
t_back_forth = 110 # s
# t_dead_el_change = 90 # s
t_dead_el_change = 10 # s

el_start = 60 # 40, 60
delta_el = -20 # deg
# delta_el = 15 # deg
az_range = 50 # deg

nb_az_scans_between_el_shifts = 8#10 # nb of back of forth scans at cst el

In [ ]:
max_nb_az_scans = duration_max/t_back_forth
steps_of_1_deg = abs(delta_el) + 1
nb_az_scans_per_1_deg_el_step = (duration_max - steps_of_1_deg*(t_dead_el_change - 1))/t_back_forth/steps_of_1_deg
print(nb_az_scans_per_1_deg_el_step)
nb_az_scans_total = duration_max/t_back_forth # if t_dead_el_change == 0

In [ ]:
time_constant_el = (t_back_forth * nb_az_scans_between_el_shifts + t_dead_el_change)
nb_el_steps_max = duration_max/time_constant_el

In [ ]:
delta_el_change_max = delta_el/nb_el_steps_max

In [ ]:
print("duration max of the observation is {} seconds, which is {} hours".format(duration_max, duration_max/3600))
print("elevation steps of {} degrees".format(delta_el_change_max))
print("nb max of el steps is {}".format(nb_el_steps_max))
print("we start at el = {} and end max at el = {}".format(el_start, el_start + delta_el))

In [ ]:
nb_el_steps = int(nb_el_steps_max)
nb_el_change = (nb_el_steps - 1)
precision_el_change = 1/10 # deg # 1/4 # should be greater than 1/100
if precision_el_change<=1/100:
    raise ValueError(r"precision_el_change is lower or equal to 1/100, which can lead to issue later with the {:.2f} value print")
delta_el_change = int(delta_el/nb_el_change/precision_el_change)*precision_el_change
duration = nb_el_steps * time_constant_el #- t_dead_el_change # we keep this time in the scanning strategy for now
h_start = t_start["h"]
m_start = t_start["m"]
t_end = h_start*3600 + m_start*60 + duration
h_end = (t_end//3600)%24
m_end = (t_end%3600)//60
s_end = (t_end%3600)%60
el_end = el_start + nb_el_change*delta_el_change
print("Total duration of observation: {} seconds".format(duration))
print("It starts at {}:{} and ends at {}:{}:{} for {} elevation steps".format(h_start, m_start, h_end, m_end, s_end, nb_el_steps))
# print("{} back and forth azimuth scans".format(nb_az_scans_total))
print("{} back and forth azimuth scans per elevation step".format(nb_az_scans_between_el_shifts))
print("We start at el = {} deg and end at el = {} deg with elevation changes of {} deg".format(el_start, el_end, delta_el_change))

In [ ]:
print(r"```")
print("time_start = {:02d}:{:02d} UTC".format(h_start, m_start))
print("full_duration = {} # sec, i.e. {}h{}m{}s".format(duration, duration//3600, (duration%3600)//60, (duration%3600)%60))
print("az_range = {} # deg".format(az_range))
print("el_start = {} #deg".format(el_start))
print("el_step = {:.2f} # deg".format(delta_el_change))
print("duration_fixed_el = {} # sec, i.e. {} back and forth scans".format(time_constant_el, nb_az_scans_between_el_shifts))
print(r"```")
print("Estimated end time = {:02d}:{:02d}:{:02d}".format(h_end, m_end, s_end))
print("End elevation = {:.2f} deg".format(el_end))